### 01 — Exploratory Data Analysis
**Dataset:** Give Me Some Credit (`cs-training.csv`)
**Target:** `SeriousDlqin2yrs` — 90+ days past due in the next 2 years (1 = default)

Goals of this notebook:
1. Load and validate the raw data (shape, dtypes)
2. Target balance
3. Missing values
4. Distributions, outliers and data-quality anomalies
5. Feature vs. target relationships
6. Decisions to carry into `02_baseline`

**1 - Create Schema and Volume**

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.risco_credito;
CREATE VOLUME IF NOT EXISTS workspace.risco_credito.raw;

**3 - Test**

In [0]:
import pandas as pd

PATH = "/Volumes/workspace/risco_credito/raw/cs-training.csv"
df = pd.read_csv(PATH, index_col=0)

print(df.shape)
df.head()

In [0]:
%pip install seaborn


In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

**4 - Parameters**

In [0]:
pd.set_option("display.max_columns", None)                 # show all columns when displaying a DataFrame (pandas hides some with "..." by default)
pd.set_option("display.float_format", "{:,.4f}".format)    # 4 decimals with thousands separator instead of scientific notation (1.2e+05)
sns.set_theme(style="whitegrid")                           # default plot style
 
DATA_PATH = "/Volumes/workspace/risco_credito/raw/cs-training.csv"  # file path in the Unity Catalog Volume
TARGET = "SeriousDlqin2yrs"                                          # column we want to predict


In [0]:
df = pd.read_csv(DATA_PATH, index_col=0)   # read the CSV into a DataFrame
 
assert df.shape == (150_000, 11), f"Unexpected shape: {df.shape}"   # (rows, columns) — expect 150k borrowers and 11 columns
print(df.shape)
df.info()

In [0]:
df.head() #show first 5 rows

- Each line: one client
- `SeriousDlqin2yrs` is what happened *afterwards* (1 = became 90+ days past due within 2 years).



In [0]:
print("Duplicated rows:", df.duplicated().sum())

**5 - Target balance**

In [0]:
target_counts = df[TARGET].value_counts()   # count of 0s and 1s
target_rate = df[TARGET].mean()             # mean of 0/1 = share of 1s = default rate
print(target_counts)
print(f"Default rate: {target_rate:.2%}")   # :.2% formats as a percentage with 2 decimals
 
ax = target_counts.plot(kind="bar", title="Target distribution")
ax.set_xlabel(TARGET); ax.set_ylabel("count")
plt.show()

- Databricks visualization:

In [0]:
# Summary table: one row per class, with count and share of borrowers
target_summary = (
    df[TARGET]
      .value_counts()
      .rename_axis("class")              # index name -> becomes a column
      .reset_index(name="count")         # turn the Series into a 2-column table
      .assign(
          label=lambda x: x["class"].map({0: "No default (0)", 1: "Default (1)"}),  # readable labels for the chart
          pct=lambda x: x["count"] / x["count"].sum(),                                # share of total
      )
      .sort_values("class")
)

target_rate = df[TARGET].mean()           # still needed later (red reference line in section 5)
print(f"Default rate: {target_rate:.2%}")

display(target_summary)                   # Databricks interactive table -> add a visualization from it

Databricks visualization. Run in Databricks to view.

**6 - Missing Values**

In [0]:
missing = (
    df.isna().sum()
      .to_frame("n_missing")
      .assign(pct=lambda x: x["n_missing"] / len(df))
      .query("n_missing > 0")
      .sort_values("n_missing", ascending=False)
)
missing

In [0]:
for col in missing.index:
    rates = df.groupby(df[col].isna())[TARGET].mean() # splits borrowers into two groups (False = present, True = missing) and  computes each group's default rate.
    print(f"{col}: default rate | present={rates.get(False, np.nan):.2%} | missing={rates.get(True, np.nan):.2%}")

**7. Distributions, outliers & anomalies**

In [0]:
df.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

In [0]:
features = [c for c in df.columns if c != TARGET]   # every column except the target
 
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
for ax, col in zip(axes.flat, features):            # zip pairs each grid axis with a feature
    s = df[col].dropna()                             # drop NaN (histograms can't handle them)
    s.clip(upper=s.quantile(0.99)).hist(bins=50, ax=ax)
    ax.set_title(col, fontsize=9)
for ax in axes.flat[len(features):]:                 # hide unused axes
    ax.set_visible(False)
plt.tight_layout()                                   # adjust spacing so titles don't overlap
plt.show()

In [0]:
past_due_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",   # times 30–59 days past due
    "NumberOfTime60-89DaysPastDueNotWorse",   # times 60–89 days past due
    "NumberOfTimes90DaysLate",                # times 90+ days past due
]
 
print("age == 0:", (df["age"] == 0).sum())
print("age < 18:", (df["age"] < 18).sum())
print("RevolvingUtilization > 1:", (df["RevolvingUtilizationOfUnsecuredLines"] > 1).sum())
print("RevolvingUtilization > 10:", (df["RevolvingUtilizationOfUnsecuredLines"] > 10).sum())
 
for col in past_due_cols:
    print(f"{col} — largest values:")
    print(df[col].value_counts().sort_index().tail(5), "\n")

In [0]:
sentinel = df[past_due_cols].isin([96, 98]).any(axis=1)
print("Rows with 96/98 in any past-due column:", sentinel.sum())
print(f"Default rate in those rows: {df.loc[sentinel, TARGET].mean():.2%}")

In [0]:
df.groupby(df["MonthlyIncome"].isna())["DebtRatio"].describe(percentiles=[.5, .95, .99])